# M2 embedding extraction checks (PLAN §7 M2; DONE DC-06, DC-09, DC-10, DC-12)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**
(and **GITHUB_TOKEN** only if the repo is private). **Restart the kernel** before *Run all*.

Expected wall-clock ≈ 60–80 min, most of it downloading Llama-3.1-8B (16 GB) and Gemma-3-4B
(8.6 GB). Paste back the output of the last cell (and any cell that fails).

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = dict(os.environ)
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[train,dev]" 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze_m2.txt
# Preflight (B-012): fails here, not inside a stage, if torch and torchvision do not match.
!python -c "import torch, torchvision, transformers; from transformers import Qwen2ForCausalLM, LlamaForCausalLM, Gemma3ForConditionalGeneration; print('preflight ok', torch.__version__, torchvision.__version__, transformers.__version__)"

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# M2-T3 Tier 0: Qwen2.5-0.5B reference extraction on the smoke subset, then the identical run
# again. Expected: first "ran", second "cache hit" for both stages (DC-10).
!python -m rbbd.cli run --config configs/smoke.yaml --stages sentences,embed
!python -m rbbd.cli run --config configs/smoke.yaml --stages sentences,embed

In [ ]:
# DC-06 (D-061: fp32 exact + fp16 per-sentence cosine) and DC-09 (guard on a real forward) on the T4.
!pytest -q -m gpu -rfE --tb=short tests/gpu/test_extract_gpu.py 2>&1 | tee /kaggle/working/artifacts/env/$RBBD_SESSION_ID/pytest_gpu_extract.txt | tail -40

In [ ]:
# M2-T4 DC-12: Llama-3.1-8B fp16 sharded over both T4s, full sentence union, post- and
# pre-norm sites. timing.json separates download, load and extraction seconds.
!python -m rbbd.cli run --config configs/tier1_llama3.1-8b.yaml --stages sentences,embed

In [ ]:
# M2-T4 Gemma probe (B-001, D-005): fp16 first (the guard may fire; that is a result, not a
# notebook error), then the fp32 reference, then the comparison - only if fp16 completed (D-062).
!python -m rbbd.cli run --config configs/m2_gemma3-4b_fp16_probe.yaml --stages sentences,embed; echo $? > /tmp/rbbd_gemma_fp16_rc
!python -m rbbd.cli run --config configs/tier1_gemma3-4b.yaml --stages sentences,embed
!if [ "$(cat /tmp/rbbd_gemma_fp16_rc)" = "0" ]; then python -m rbbd.cli compare-embeddings --a configs/m2_gemma3-4b_fp16_probe.yaml --b configs/tier1_gemma3-4b.yaml; else echo "Gemma fp16 did not complete: D-058 criterion 1 fails, Gemma stays fp32 (D-062); comparison skipped"; fi

In [ ]:
# Mirror this session's outputs into the private store (D-041).
for _path in ["manifests", "sentences", "embed", "embeddings", f"env/{RBBD_SESSION_ID}"]:
    if os.path.isdir(f"/kaggle/working/artifacts/{_path}"):
        !python -m rbbd.cli sync --path {_path}

In [ ]:
# Summary to paste back.
import glob, json
for path in sorted(glob.glob("/kaggle/working/artifacts/embed/*/timing.json")):
    print(path); print(open(path).read())
for path in sorted(glob.glob(f"/kaggle/working/artifacts/env/{RBBD_SESSION_ID}/*.json")):
    if "env.json" not in path:
        print(path); print(open(path).read())
!python -m rbbd.cli status --config configs/tier1_llama3.1-8b.yaml